In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# 1. Load dataset
df = pd.read_csv("Churn_Modelling.csv")

# 2. Preprocessing
X = df.drop(["RowNumber", "CustomerId", "Surname", "Exited"], axis=1)
y = df["Exited"]
X = pd.get_dummies(X, columns=["Geography", "Gender"], drop_first=True).astype(float)

# 3. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. Feature scaling
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

# 5. Build neural network
model = Sequential([
    Dense(16, activation="relu", input_shape=(X_train.shape[1],)),
    Dropout(0.2),
    Dense(8, activation="relu"),
    Dense(1, activation="sigmoid")
])

# 6. Compile model
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# 7. Early stopping to help reduce overfitting
stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

# 8. Train model
model.fit(X_train, y_train, epochs=100, batch_size=32,
          validation_split=0.1, callbacks=[stop])

# 9. Prediction
y_pred = (model.predict(X_test) > 0.5).astype(int)

# 10. Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
